In [4]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn import datasets

In [5]:
df = datasets.load_diabetes(as_frame = True).frame

In [6]:
df.head()
df.shape

(442, 11)

In [7]:
X = df.drop("target", axis = 1)
y = df["target"]

In [8]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [10]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).ravel()

In [11]:
# Model
model = SVR()

model.fit(X_train, y_train_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [12]:
y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)

In [13]:
print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: 0.6596361676267712
test r2: 0.48844443151651906


In [14]:
# Linear
model = SVR(kernel = "linear")
model.fit(X_train, y_train_scaled)

y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)

print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: 0.45191229982475245
test r2: 0.4433761323833776


In [15]:
# Linear
model = SVR(kernel = "sigmoid")

model.fit(X_train, y_train_scaled)

y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)

print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: -19.72119344073131
test r2: -15.316808189576822


In [16]:
# Linear
model = SVR(kernel = "poly")

model.fit(X_train, y_train_scaled)

y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)

print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: 0.5790920834310542
test r2: 0.24203771038107735


## Hyperparameter tuning using GridSearchCV

In [25]:
from sklearn.model_selection import GridSearchCV

In [26]:
param_grid = {
    "C" : [1,2,5, 10, 50, 100],
    "kernel" : ["rbi", "linear"],
    "epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [27]:
svr = SVR()

grid_search = GridSearchCV(svr, param_grid, scoring = "r2", cv = 5)

grid_search.fit(X_train, y_train_scaled)

C:\Users\alekh\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
150 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\alekh\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alekh\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py", line 1358, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\alekh\AppData\Roaming\Pyth

,estimator,SVR()
,param_grid,"{'C': [1, 2, ...], 'epsilon': [0.01, 0.1, ...], 'kernel': ['rbi', 'linear']}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'linear'


In [28]:
print("best params - ", grid_search.best_params_)

best params -  {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [29]:
best_model = SVR(kernel = "linear", C = 10, epsilon = 0.1)

best_model.fit(X_train, y_train_scaled)

y_train_pred_scaled = best_model.predict(X_train)
y_test_pred_scaled = best_model.predict(X_test)

print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: 0.5151066486918875
test r2: 0.47444183250401095


In [31]:
from sklearn.svm import LinearSVR

model = LinearSVR(C = 10, epsilon = 0.1, max_iter = 1000)

model.fit(X_train, y_train_scaled)

y_train_pred_scaled = model.predict(X_train)
y_test_pred_scaled = model.predict(X_test)

print("train r2:", r2_score(y_train_scaled, y_train_pred_scaled))
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))

train r2: 0.5150383461606935
test r2: 0.4742702149559065


C:\Users\alekh\AppData\Roaming\Python\Python313\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
